In [1]:
import json
import logging
import os
from pathlib import Path

import boto3
import ipynbname
import numpy as np
import pandas as pd
from dotenv import load_dotenv

In [2]:
p = ipynbname.path().parent.parent

format = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
filename_path = Path(p / 'logs' / 'transform_data.log')
logging.basicConfig(format=format, filename=filename_path, encoding='utf-8', level=logging.INFO)
logger = logging.getLogger(__name__)

load_dotenv(dotenv_path=p / '.env')

def extract_data():
    if not os.getenv('API_KEY'):
        logger.error("API_KEY is not set in the environment variables.")
        raise TypeError("Expected string for API_KEY, received None (API_KEY is not set in the environment variables).")

    try:
        s3_client = boto3.client(
            's3', 
            aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'), 
            aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
        )
        response = s3_client.get_object(
            Bucket=os.getenv('S3_BUCKET_NAME'), 
            Key=f"bronze/mercadolivre/celulares-e-telefones/2026-10-18 10:24:00.662699-03:00/bronze_data.json"
        )
        data = json.loads(response['Body'].read().decode('utf-8'))
        logger.info("bronze_data.json obtained from Amazon S3")
        return data
    except Exception as e:
        logger.error(f"Error occurred while extracting data from Amazon S3: {e}")
        raise

In [3]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

'''def create_dataframe() -> pd.DataFrame:
    data = extract_data()
    df = pd.json_normalize(data)
    return df

df = create_dataframe()'''

with open(p / "bronze_data.json", encoding="utf-8") as f:
    obj = json.load(f)
    df = pd.json_normalize(obj)

# 1

In [4]:
def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    df_data_items = df.explode('data.items', ignore_index=True)
    df_items = pd.json_normalize(df_data_items['data.items'])
    df = pd.concat(
        [df_data_items, df_items],
        axis=1
    )
    return df

df = normalize_df(df)
df.head()

,requestId,executionId,data.source,data.type,data.url,data.requestUrl,data.extractedAt,data.query,data.totalResults,data.primaryResults,data.page,data.resultsPerPage,data.offset,data.nextPage,data.nextPageUrl,data.items,url,imageUrl,sku,categoryId,domainId,name,condition,currency,currencyRaw,price,highlight,isBestSeller,isPowerSeller,powerSellerStatusTitle,ean,sellerId,sellerName,sellerCity,sellerState,sellerCountry,aggregateRating.rating,aggregateRating.reviewCount,aggregateRating
0,d40ebab3-b318-4474-8cab-b9b1b3d4c60d,2701b280-67e5-4dff-a23e-57890a384713,mercadolivre.com.br,plp,https://lista.mercadolivre.com.br/iphone-15-pro-max,https://lista.mercadolivre.com.br/iphone-15-pro-max,2026-10-20T14:36:11.749Z,iphone 15 pro max,196,196,1,48,0,2.0,https://lista.mercadolivre.com.br/celulares-telefones/celulares-smartphones/iphone/iphone-15-pro-max/iphone-15-pro-max_Desde_49_NoIndex_True,"{'url': 'https://www.mercadolivre.com.br/apple-iphone-15-pro-max-256-gb-titanio-natural-excelente-recondicionado/p/MLB2000122364', 'imageUrl': 'https://http2.mlstatic.com/D_Q_NP_2X_729795-MLA95495293322_102025-E.webp', 'sku': 'MLB5379973109', 'categoryId': 'MLB1055', 'domainId': 'MLB-CELLPHONES', 'name': 'Apple iPhone 15 Pro Max (256 GB) - Titânio Natural - Excelente (Recondicionado)', 'condition': 'Recondicionado', 'currency': 'BRL', 'currencyRaw': 'R$', 'price': 4299, 'highlight': 'ÚLTIMAS 4', 'isBestSeller': False, 'isPowerSeller': False, 'powerSellerStatusTitle': None, 'aggregateRating': {'rating': 4.6, 'reviewCount': None}, 'ean': None, 'sellerId': None, 'sellerName': None, 'sellerCity': None, 'sellerState': None, 'sellerCountry': None}",https://www.mercadolivre.com.br/apple-iphone-15-pro-max-256-gb-titanio-natural-excelente-recondicionado/p/MLB2000122364,https://http2.mlstatic.com/D_Q_NP_2X_729795-MLA95495293322_102025-E.webp,MLB5379973109,MLB1055,MLB-CELLPHONES,Apple iPhone 15 Pro Max (256 GB) - Titânio Natural - Excelente (Recondicionado),Recondicionado,BRL,R$,4299,ÚLTIMAS 4,False,False,None,None,None,NaN,None,None,None,4.6,NaN,NaN
1,d40ebab3-b318-4474-8cab-b9b1b3d4c60d,2701b280-67e5-4dff-a23e-57890a384713,mercadolivre.com.br,plp,https://lista.mercadolivre.com.br/iphone-15-pro-max,https://lista.mercadolivre.com.br/iphone-15-pro-max,2026-10-20T14:36:11.749Z,iphone 15 pro max,196,196,1,48,0,2.0,https://lista.mercadolivre.com.br/celulares-telefones/celulares-smartphones/iphone/iphone-15-pro-max/iphone-15-pro-max_Desde_49_NoIndex_True,"{'url': 'https://www.mercadolivre.com.br/apple-iphone-15-pro-max-1-tb-titanio-natural-excelente-recondicionado/p/MLB2000019753', 'imageUrl': 'https://http2.mlstatic.com/D_Q_NP_2X_774199-MLA95936608785_102025-E.webp', 'sku': 'MLB7842881974', 'categoryId': 'MLB1055', 'domainId': 'MLB-CELLPHONES', 'name': 'Apple iPhone 15 Pro Max (1 TB) - Titânio Natural - Excelente (Recondicionado)', 'condition': 'Recondicionado', 'currency': 'BRL', 'currencyRaw': 'R$', 'price': 5989, 'highlight': 'ÚLTIMA!', 'isBestSeller': False, 'isPowerSeller': False, 'powerSellerStatusTitle': None, 'aggregateRating': {'rating': 4.6, 'reviewCount': None}, 'ean': None, 'sellerId': None, 'sellerName': 'Mobi Phone', 'sellerCity': None, 'sellerState': None, 'sellerCountry': None}",https://www.mercadolivre.com.br/apple-iphone-15-pro-max-1-tb-titanio-natural-excelente-recondicionado/p/MLB2000019753,https://http2.mlstatic.com/D_Q_NP_2X_774199-MLA95936608785_102025-E.webp,MLB7842881974,MLB1055,MLB-CELLPHONES,Apple iPhone 15 Pro Max (1 TB) - Titânio Natural - Excelente (Recondicionado),Recondicionado,BRL,R$,5989,ÚLTIMA!,False,False,None,None,None,Mobi Phone,None,None,None,4.6,NaN,NaN
2,d40ebab3-b318-4474-8cab-b9b1b3d4c60d,2701b280-67e5-4dff-a23e-57890a384713,mercadolivre.com.br,plp,https://lista.mercadolivre.com.br/iphone-15-pro-max,https://lista.mercadolivre.com.br/iphone-15-pro-max,2026-10-20T14:36:11.749Z,iphone 15 pro max,196,196,1,48,0,2.0,https://lista.mercadolivre.com.br/celulares-telefones/celulares-smartphones/iphone/iphone-15-pro-max/iphone-15-p

# 2

In [5]:
drop_columns = [
    'requestId', 'executionId', 'data.source', 'data.type', 'data.url', 
    'data.requestUrl', 'data.query', 'data.totalResults', 'data.primaryResults', 
    'data.page', 'data.resultsPerPage', 'data.offset', 'data.nextPage', 
    'data.nextPageUrl', 'data.items', 'aggregateRating'
]

def drop_unnecessary(df: pd.DataFrame, drop_columns: list) -> pd.DataFrame:
    df = df.drop(drop_columns, axis=1)
    return df

df = drop_unnecessary(df, drop_columns)
resume = pd.DataFrame({
    'dtype': df.dtypes,
    'total': len(df),
    'not_null': df.notna().sum(),
    'nulls': df.isna().sum(),
    'unique': df.nunique()
})
print(resume)

                               dtype  total  not_null  nulls  unique
data.extractedAt                 str    196       196      0       5
url                              str    196       196      0     190
imageUrl                         str    196       196      0     188
sku                              str    196       196      0     190
categoryId                       str    196       195      1       1
domainId                         str    196       196      0       1
name                             str    196       196      0     124
condition                        str    196       192      4       3
currency                         str    196       196      0       1
currencyRaw                      str    196       196      0       1
price                          int64    196       196      0     129
highlight                        str    196        11    185       5
isBestSeller                    bool    196       196      0       1
isPowerSeller                   bo

# 3

In [6]:
columns_to_rename = {
    'url': 'url',
    'sku': 'sku',
    'data.extractedAt': 'extracted_at',
    'imageUrl': 'image_url',
    'categoryId': 'category_id',
    'domainId': 'domain_id',
    'name': 'name',
    'condition': 'condition',
    'currency': 'currency',
    'currencyRaw': 'currency_raw',
    'price': 'price',
    'highlight': 'highlight',
    'isBestSeller': 'is_best_seller',
    'isPowerSeller': 'is_power_seller',
    'powerSellerStatusTitle': 'power_seller_status_title',
    'ean': 'ean',
    'sellerId': 'seller_id',
    'sellerName': 'seller_name',
    'sellerCity': 'seller_city',
    'sellerState': 'seller_state',
    'sellerCountry': 'seller_country',
    'aggregateRating.rating': 'rating',
    'aggregateRating.reviewCount': 'review_count',
}

def rename_columns(df: pd.DataFrame, columns_to_rename: dict) -> pd.DataFrame:
    for key, value in columns_to_rename.items():
        df.rename(columns={key: value}, inplace=True)

    return df

df = rename_columns(df, columns_to_rename)
print(df.dtypes)


extracted_at                     str
url                              str
image_url                        str
sku                              str
category_id                      str
domain_id                        str
name                             str
condition                        str
currency                         str
currency_raw                     str
price                          int64
highlight                        str
is_best_seller                  bool
is_power_seller                 bool
power_seller_status_title     object
ean                           object
seller_id                     object
seller_name                      str
seller_city                   object
seller_state                  object
seller_country                object
rating                       float64
review_count                 float64
dtype: object


# 4

In [7]:
df.head()

,extracted_at,url,image_url,sku,category_id,domain_id,name,condition,currency,currency_raw,price,highlight,is_best_seller,is_power_seller,power_seller_status_title,ean,seller_id,seller_name,seller_city,seller_state,seller_country,rating,review_count
0,2026-10-20T14:36:11.749Z,https://www.mercadolivre.com.br/apple-iphone-15-pro-max-256-gb-titanio-natural-excelente-recondicionado/p/MLB2000122364,https://http2.mlstatic.com/D_Q_NP_2X_729795-MLA95495293322_102025-E.webp,MLB5379973109,MLB1055,MLB-CELLPHONES,Apple iPhone 15 Pro Max (256 GB) - Titânio Natural - Excelente (Recondicionado),Recondicionado,BRL,R$,4299,ÚLTIMAS 4,False,False,None,None,None,NaN,None,None,None,4.6,NaN
1,2026-10-20T14:36:11.749Z,https://www.mercadolivre.com.br/apple-iphone-15-pro-max-1-tb-titanio-natural-excelente-recondicionado/p/MLB2000019753,https://http2.mlstatic.com/D_Q_NP_2X_774199-MLA95936608785_102025-E.webp,MLB7842881974,MLB1055,MLB-CELLPHONES,Apple iPhone 15 Pro Max (1 TB) - Titânio Natural - Excelente (Recondicionado),Recondicionado,BRL,R$,5989,ÚLTIMA!,False,False,None,None,None,Mobi Phone,None,None,None,4.6,NaN
2,2026-10-20T14:36:11.749Z,https://www.mercadolivre.com.br/apple-iphone-15-pro-max-256-gb-titanio-natural-novo-com-caixa-aberta/p/MLB2016209990,https://http2.mlstatic.com/D_Q_NP_2X_729795-MLA95495293322_102025-E.webp,MLB7523362188,MLB1055,MLB-CELLPHONES,Apple iPhone 15 Pro Max (256 GB) - Titânio Natural (Novo com caixa aberta),Caixa aberta,BRL,R$,5124,ÚLTIMAS 3,False,False,None,None,None,NaN,None,None,None,4.0,NaN
3,2026-10-20T14:36:11.749Z,https://www.mercadolivre.com.br/apple-iphone-15-pro-max-256gb-titanio-natural-5g-original/up/MLBU3952749894,https://http2.mlstatic.com/D_Q_NP_2X_931212-MLB117528231503_092026-E.webp,MLB6731711602,MLB1055,MLB-CELLPHONES,Apple iPhone 15 Pro Max 256gb Titânio Natural 5g Original (Novo com caixa aberta),Caixa aberta,BRL,R$,5484,NaN,False,False,None,None,None,NaN,None,None,None,4.6,NaN
4,2026-10-20T14:36:11.749Z,https://www.mercadolivre.com.br/apple-iphone-15-pro-max-512-gb-titanio-azul-aceitavel-recondicionado/p/MLB2000116468,https://http2.mlstatic.com/D_Q_NP_2X_688773-MLA95494114970_102025-E.webp,MLB5425278273,MLB1055,MLB-CELLPHONES,Apple iPhone 15 Pro Max (512 GB) - Titânio Azul - Aceitável (Recondicionado),Recondicionado,BRL,R$,4369,NaN,False,False,None,None,None,NaN,None,None,None,NaN,NaN


In [8]:
def convert_type(df: pd.DataFrame) -> pd.DataFrame:
    df['extracted_at'] = pd.to_datetime(df['extracted_at'], errors='coerce').dt.tz_convert('America/Sao_Paulo')
    df = df.astype(
            {
                "url": "string", 
                "image_url": "string", 
                "sku": "string",
                "category_id": "string",
                "domain_id": "string",
                "name": "string",
                "condition": "string",
                "currency": "string",
                "currency_raw": "string",
                "price": "Float64",
                "highlight": "string",
                "is_best_seller": "boolean",
                "is_power_seller": "boolean",
                "power_seller_status_title": "string",
                "rating": "Float64",
                "review_count": "Float64",
                "ean": "string",
                "seller_id": "string",
                "seller_name": "string",
                "seller_city": "string",
                "seller_state": "string",
                "seller_country": "string"
            }
        )

    return df

df = convert_type(df)
df.head()

,extracted_at,url,image_url,sku,category_id,domain_id,name,condition,currency,currency_raw,price,highlight,is_best_seller,is_power_seller,power_seller_status_title,ean,seller_id,seller_name,seller_city,seller_state,seller_country,rating,review_count
0,2026-10-20 11:36:11.749000-03:00,https://www.mercadolivre.com.br/apple-iphone-15-pro-max-256-gb-titanio-natural-excelente-recondicionado/p/MLB2000122364,https://http2.mlstatic.com/D_Q_NP_2X_729795-MLA95495293322_102025-E.webp,MLB5379973109,MLB1055,MLB-CELLPHONES,Apple iPhone 15 Pro Max (256 GB) - Titânio Natural - Excelente (Recondicionado),Recondicionado,BRL,R$,4299.0,ÚLTIMAS 4,False,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4.6,<NA>
1,2026-10-20 11:36:11.749000-03:00,https://www.mercadolivre.com.br/apple-iphone-15-pro-max-1-tb-titanio-natural-excelente-recondicionado/p/MLB2000019753,https://http2.mlstatic.com/D_Q_NP_2X_774199-MLA95936608785_102025-E.webp,MLB7842881974,MLB1055,MLB-CELLPHONES,Apple iPhone 15 Pro Max (1 TB) - Titânio Natural - Excelente (Recondicionado),Recondicionado,BRL,R$,5989.0,ÚLTIMA!,False,False,<NA>,<NA>,<NA>,Mobi Phone,<NA>,<NA>,<NA>,4.6,<NA>
2,2026-10-20 11:36:11.749000-03:00,https://www.mercadolivre.com.br/apple-iphone-15-pro-max-256-gb-titanio-natural-novo-com-caixa-aberta/p/MLB2016209990,https://http2.mlstatic.com/D_Q_NP_2X_729795-MLA95495293322_102025-E.webp,MLB7523362188,MLB1055,MLB-CELLPHONES,Apple iPhone 15 Pro Max (256 GB) - Titânio Natural (Novo com caixa aberta),Caixa aberta,BRL,R$,5124.0,ÚLTIMAS 3,False,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4.0,<NA>
3,2026-10-20 11:36:11.749000-03:00,https://www.mercadolivre.com.br/apple-iphone-15-pro-max-256gb-titanio-natural-5g-original/up/MLBU3952749894,https://http2.mlstatic.com/D_Q_NP_2X_931212-MLB117528231503_092026-E.webp,MLB6731711602,MLB1055,MLB-CELLPHONES,Apple iPhone 15 Pro Max 256gb Titânio Natural 5g Original (Novo com caixa aberta),Caixa aberta,BRL,R$,5484.0,<NA>,False,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4.6,<NA>
4,2026-10-20 11:36:11.749000-03:00,https://www.mercadolivre.com.br/apple-iphone-15-pro-max-512-gb-titanio-azul-aceitavel-recondicionado/p/MLB2000116468,https://http2.mlstatic.com/D_Q_NP_2X_688773-MLA95494114970_102025-E.webp,MLB5425278273,MLB1055,MLB-CELLPHONES,Apple iPhone 15 Pro Max (512 GB) - Titânio Azul - Aceitável (Recondicionado),Recondicionado,BRL,R$,4369.0,<NA>,False,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [9]:
print(df.dtypes)

extracted_at                 datetime64[us, America/Sao_Paulo]
url                                                     string
image_url                                               string
sku                                                     string
category_id                                             string
domain_id                                               string
name                                                    string
condition                                               string
currency                                                string
currency_raw                                            string
price                                                  Float64
highlight                                               string
is_best_seller                                         boolean
is_power_seller                                        boolean
power_seller_status_title                               string
ean                                                    

# 5

In [10]:
print(df.loc[df['price'].isnull(), 'price'].tolist())
print(df.loc[(df['rating'] < 1) | (df['rating'] > 5), 'price'].tolist())

def treat_outliers(df: pd.DataFrame) -> pd.DataFrame:
    #price
    df = df.dropna(subset='price', ignore_index=True)
    #rating
    df.loc[(df['rating'] < 1) | (df['rating'] > 5), 'rating'] = 0

    return df

print(df.loc[df['price'].isnull(), 'price'].tolist())
print(df.loc[(df['rating'] < 1) | (df['rating'] > 5), 'price'].tolist())


[]
[]
[]
[]


# 6

In [11]:
unique_values = pd.DataFrame(df['name'].unique(), columns=['name'])
unique_values

,name
0,Apple iPhone 15 Pro Max (256 GB) - Titânio Natural - Excelente (Recondicionado)
1,Apple iPhone 15 Pro Max (1 TB) - Titânio Natural - Excelente (Recondicionado)
2,Apple iPhone 15 Pro Max (256 GB) - Titânio Natural (Novo com caixa aberta)
3,Apple iPhone 15 Pro Max 256gb Titânio Natural 5g Original (Novo com caixa aberta)
4,Apple iPhone 15 Pro Max (512 GB) - Titânio Azul - Aceitável (Recondicionado)
5,Apple iPhone 15 Pro Max (512 GB) - Titânio Natural (Novo com caixa aberta)
6,Apple iPhone 15 Pro Max (256 Gb) - Titânio Preto
7,iPhone 15 Pro Max 256gb Natural 5g Versão Brasil Anatel (Novo com caixa aberta)
8,Apple iPhone 15 Pro Max (256 GB) - Titânio Preto
9,iPhone 15 Pro Max Preto (512gb) Oferta Envio Imediato 5g (Novo com caixa aberta)


# 7

In [12]:
resume = pd.DataFrame({
    'total': len(df),
    'nulls': df.isna().sum(),
    'unique': df.nunique()
})
print(resume)

def treat_identification_columns(df: pd.DataFrame) -> pd.DataFrame:
    #unique
    df.drop_duplicates(subset=['url', 'sku'], inplace=True, ignore_index=True)
    other_columns = [col for col in df.columns if col not in ['url', 'sku']]
    df = df.reindex(columns=['url', 'sku'] + other_columns)
    
    return df

df = treat_identification_columns(df)

resume = pd.DataFrame({
    'total': len(df),
    'nulls': df.isna().sum(),
    'unique': df.nunique()
})
print(f"\n{resume}")

                           total  nulls  unique
extracted_at                 196      0       5
url                          196      0     190
image_url                    196      0     188
sku                          196      0     190
category_id                  196      1       1
domain_id                    196      0       1
name                         196      0     124
condition                    196      4       3
currency                     196      0       1
currency_raw                 196      0       1
price                        196      0     129
highlight                    196    185       5
is_best_seller               196      0       1
is_power_seller              196      0       1
power_seller_status_title    196    196       0
ean                          196    196       0
seller_id                    196    196       0
seller_name                  196    186       3
seller_city                  196    196       0
seller_state                 196    196 